# 图像卷积

## 互相关运算

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    '''计算二维互相关运算'''
    h, w = K.shape
    Y = torch.zeros(X.shape[0]-h+1, X.shape[1]-w+1) # 输出形状
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w]*K).sum() # 计算互相关
    return Y

### 验证上述二维互相关运算的输出

In [3]:
X = torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8]])
K = torch.tensor([[0, 1], [2, 3]])
Y = corr2d(X, K)
print(Y)

tensor([[19., 25.],
        [37., 43.]])


## 实现二维卷积层

In [ ]:
class Conv2D(nn.Module):
    '''二维卷积层'''
    def __init__(self, kernel_size):
        super(Conv2D, self).__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size)) # torch.rand(kernel_size)为一个长为kernel_size的向量，不是长宽都是kernel_size的矩阵
        self.bias = nn.Parameter(torch.rand(1))

    def forward(self, X):
        return corr2d(X, self.weight) + self.bias

### 卷积层的一个简单应用

In [4]:
X = torch.ones(6, 8) # 输入形状为(6, 8)
X[:, 2:6] = 0 # 在X的第2到第5列设置为0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [5]:
K = torch.tensor([[1, -1]]) # 卷积核

### 输出Y中的1代表从白色到黑色的边缘，-1代表黑色到白色的边缘

In [6]:
Y = corr2d(X, K) # 计算互相关
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

### 卷积核K只能检测垂直边缘

In [10]:
print(corr2d(X.T, K))
print(corr2d(X, K.T))
print(corr2d(X.T, K.T)) # 输出形状

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 1.,  1.,  1.,  1.,  1.,  1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.],
        [-1., -1., -1., -1., -1., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.]])


## 学习由X生成Y的卷积核

In [21]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False) # 创建一个卷积层
X = X.reshape((1, 1, 6, 8)) # 调整输入形状为(1, 1, 6, 8)
Y = Y.reshape((1, 1, 6, 7)) # 调整输出形状为(1, 1, 5, 7)

for i in range(10):
    Y_hat = conv2d(X)
    loss = (Y_hat-Y)**2
    conv2d.zero_grad() # 清除梯度
    loss.sum().backward() # 计算梯度
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad # 更新权重
    if (i+1) % 2 == 0:
        print(f'epoch {i+1}, loss {loss.sum():.3f}, weight {conv2d.weight.data.reshape(1,2)}')

epoch 2, loss 3.247, weight tensor([[ 0.6353, -0.9040]])
epoch 4, loss 0.954, weight tensor([[ 0.8196, -0.9917]])
epoch 6, loss 0.328, weight tensor([[ 0.9063, -1.0164]])
epoch 8, loss 0.124, weight tensor([[ 0.9489, -1.0194]])
epoch 10, loss 0.049, weight tensor([[ 0.9710, -1.0161]])


## 练习

1. 构建一个具有对角线边缘的图像`X`。
    1. 如果将本节中举例的卷积核`K`应用于`X`，会发生什么情况？
    1. 如果转置`X`会发生什么？
    1. 如果转置`K`会发生什么？
1. 在我们创建的`Conv2D`自动求导时，有什么错误消息？
1. 如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？
1. 手工设计一些卷积核。
    1. 二阶导数的核的形式是什么？
    1. 积分的核的形式是什么？
    1. 得到$d$次导数的最小核的大小是多少？
